In [ ]:
# === Colab rescue scaffolding: mount Drive + tee stdout/stderr + checkpoints ====
# Streams every log line to Google Drive so an interrupted T4 session keeps its
# evidence. No token is hard-coded. Safe to skip if Drive is unavailable.
import os, sys, json, time, atexit, shutil
from pathlib import Path

CALM_RUN_STAMP = time.strftime("%Y%m%d_%H%M%S")
CALM_DRIVE_DIR = None
CALM_LOCAL_DIR = Path("/content") if Path("/content").exists() else Path.cwd()

class _Tee:
    def __init__(self, stream, path):
        self.stream = stream; self.file_path = Path(path)
        self.file_path.parent.mkdir(parents=True, exist_ok=True)
        self.file = open(self.file_path, "a", encoding="utf-8", buffering=1)
        self._calm_tee = True; self.encoding = getattr(stream, "encoding", "utf-8")
    def write(self, data):
        try: self.stream.write(data)
        except Exception: pass
        try: self.file.write(data); self.file.flush()
        except Exception: pass
    def flush(self):
        for t in (self.stream, self.file):
            try: t.flush()
            except Exception: pass
    def isatty(self): return False
    def __getattr__(self, name): return getattr(self.stream, name)

def _mount_drive():
    global CALM_DRIVE_DIR
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        root = Path("/content/drive/MyDrive/calm_fuzz_12b_runs")
        CALM_DRIVE_DIR = root / CALM_RUN_STAMP
        CALM_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        return CALM_DRIVE_DIR
    except Exception as e:
        CALM_DRIVE_DIR = None
        print(">> No Google Drive; keeping artifacts in the runtime only:", str(e)[:140])
        return None

def calm_local(name): return CALM_LOCAL_DIR / name
def calm_drive(name): return (Path(CALM_DRIVE_DIR) / name) if CALM_DRIVE_DIR else None
def calm_copy_to_drive(path, name=None):
    if not CALM_DRIVE_DIR: return None
    src = Path(path)
    if not src.exists(): return None
    dst = Path(CALM_DRIVE_DIR) / (name or src.name)
    dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); return str(dst)

_dd = _mount_drive()
if _dd:
    if not getattr(sys.stdout, "_calm_tee", False): sys.stdout = _Tee(sys.stdout, _dd / "stdout_stream.log")
    if not getattr(sys.stderr, "_calm_tee", False): sys.stderr = _Tee(sys.stderr, _dd / "stderr_stream.log")
    atexit.register(lambda: getattr(sys.stdout, "flush", lambda: None)())
    print(">> Rescue stream -> Drive:", _dd)
else:
    print(">> No Drive stream. Re-run this cell and allow mount if you want crash-proof logs.")


Mounted at /content/drive
>> Rescue stream -> Drive: /content/drive/MyDrive/calm_fuzz_12b_runs/20260611_092225


In [ ]:
# === Environment: deps, MOCK/torch/CUDA detection, Hugging Face login ==========
import os, sys, subprocess, importlib.util, re
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

def _has(m): return importlib.util.find_spec(m) is not None
def _ver(pkg):
    try:
        from importlib.metadata import version; return version(pkg)
    except Exception: return "0"
def _vparts(v):
    p = [int(x) for x in re.findall(r"\d+", str(v))[:3]]; return tuple(p + [0] * (3 - len(p)))
def _pip(specs):
    if specs:
        print(">> install:", " ".join(specs))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *specs], check=False)

# core
_pip([p for p in ["numpy", "pandas", "matplotlib"] if not _has(p)])

# Toggle: set FUZZ_MOCK=1 to smoke-test the whole pipeline on CPU with no model.
MOCK = os.environ.get("FUZZ_MOCK", "0") == "1"
HAVE_TORCH = _has("torch")
HAS_CUDA = False
if HAVE_TORCH and not MOCK:
    try:
        import torch; HAS_CUDA = torch.cuda.is_available()
    except Exception: HAS_CUDA = False
USE_LLM = (not MOCK) and HAS_CUDA   # Gemma only when a real GPU is present

if USE_LLM:
    specs = []
    if _vparts(_ver("transformers")) < _vparts("5.10.1"): specs.append("transformers>=5.10.1")
    for pkg, mod in [("accelerate", "accelerate"), ("bitsandbytes", "bitsandbytes"),
                     ("huggingface_hub", "huggingface_hub"), ("sentencepiece", "sentencepiece"),
                     ("protobuf", "google.protobuf"), ("safetensors", "safetensors")]:
        if not _has(mod): specs.append(pkg)
    _pip(specs)

import numpy as np
print(">> MOCK:", MOCK, "| torch:", HAVE_TORCH, "| cuda:", HAS_CUDA, "| use_llm:", USE_LLM)

# Hugging Face token from Colab secret or env (never hard-code it)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
        if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
    except Exception: pass
if HF_TOKEN and USE_LLM:
    try:
        from huggingface_hub import login; login(token=HF_TOKEN, add_to_git_credential=False)
        print(">> Logged in to Hugging Face via HF_TOKEN.")
    except Exception as e:
        print(">> HF login error:", str(e)[:120])
elif USE_LLM:
    print(">> No HF_TOKEN yet; add a Colab secret named HF_TOKEN if you hit 401/403.")


>> install: bitsandbytes
>> MOCK: False | torch: True | cuda: True | use_llm: True
>> No HF_TOKEN yet; add a Colab secret named HF_TOKEN if you hit 401/403.


In [ ]:
%%writefile calm_axiompp_v4_program_prover.py
# %% ════════════════════════════════════════════════════════════════════════
# AXIOM++ v4 — PROGRAM PROVER: máy chứng minh cho CẢ CHƯƠNG TRÌNH tensor,
#               Gemma 4 sinh CHƯƠNG TRÌNH (không phải JSON-patch), 1 file.
#
#   python calm_axiompp_v4_program_prover.py            (tự dò CPU/CUDA, tự test)
#   GEMMA_DIR=<model> python ...                        (bật Gemma 4 sinh DSL)
#   python ... --repro calm_v4_findings.json 0          (chạy lại finding số 0)
#
# QA độ-tin-cậy-số-học cho thư viện tensor (dòng TitanFuzz/NNSmith/FUEL):
# so cùng MỘT chương trình trên nhiều đường bảo-toàn-ngữ-nghĩa, chỉ lên tiếng
# khi vi phạm CHỨNG-MINH-ĐƯỢC. Báo cho maintainer, kèm repro.
#
# ── Vì sao v4 (rút từ lần chạy T4 thật + ULTIMATE) ──────────────────────────
#   • ULTIMATE chỉ chứng minh ở mức TỪNG-OP → bug fusion của torch.compile
#     (xuyên op) về cấu trúc KHÔNG với tới. v4 chứng minh cho DAG tuỳ ý.
#   • Gemma-as-JSON-patch: trả 2.6x throughput lấy ~0 thông tin (log T4:
#     burst nào cũng cùng patch, có lần "no json"). v4: Gemma sinh CHƯƠNG TRÌNH
#     theo lô (sampling, nhiều seq/lần gọi), máy chứng minh gác cổng.
#   • invalid 24% ngân sách (T4) → v4 suy luận shape TRƯỚC khi thực thi: ~0%.
#   • T4 (sm_75) không có TF32 → lớp precision-substitution không thể hiện ra,
#     "axiom 0" là ĐÚNG KỲ VỌNG. Bãi săn T4 thật: compile-fusion, CPU↔CUDA,
#     layout, reassociation, fused-vs-macro, và ĐẢO-CHÍNH-XÁC dưới autocast.
#
# ── Hai oracle hợp thành (đều là ĐỊNH LÝ, 0 báo nhầm cấu trúc) ──────────────
#   1. ĐẢO-CHÍNH-XÁC HỢP THÀNH: theo dõi cận biên độ M qua đồ thị; chừng nào
#      giá trị còn nguyên & M < 2^24 thì MỌI op fp32 đúng từng bit ⇒ cả chương
#      trình phải khớp bit trên MỌI đường (kể cả compile fuse, autocast fp16
#      khi M < 2^11, bf16 khi M < 2^8, reassociation split-k). Lệch 1 bit ⇒
#      EXACT_VIOLATION chứng minh được.
#   2. GIẢI TÍCH BAO (envelope calculus): mỗi node mang (ref_fp64, E_cert);
#      quy tắc hợp thành sai-số-thuận chuẩn (Wilkinson–Higham cho matmul/sum
#      bất kể thứ tự cộng; mul/add/exp/log/div có cận đạo-hàm-chặn). Mọi đường
#      fp32 trung thực của DAG phải có |obs − ref| ≤ E từng phần tử.
#      cert_ratio = max|Δ|/E : >1 ⇒ vi phạm chứng minh được. Xám 0.8–1.0.
#      Ghi chú trung thực: kernel fused (softmax/logsumexp) được đối chiếu với
#      bao của DAG khai triển — hợp đồng "fused chính xác ≥ bản khai triển";
#      vượt bao ⇒ ít nhất là accuracy-regression đáng báo cáo.
#   Phòng hộ: ±0/NaN chính tắc (bài học 8.897 ca rởm), tràn-số hợp lệ của fp32
#   bị loại khỏi phán xét (chỉ phán nơi |ref|+E < FLT_MAX), slack denormal.
#
# ── Gemma 4 dùng "triệt để" đúng tầng ───────────────────────────────────────
#   Model 12B sinh CHƯƠNG TRÌNH trong DSL (JSON 1 dòng), mỗi lần gọi lấy nhiều
#   chương trình (num_return_sequences, sampling) → mỗi chương trình được nhân
#   ra hàng chục instance (island/float × style giá trị × jitter shape) → một
#   lần trả phí LLM mua hàng trăm phép chứng minh. Lỗi cú pháp được ĐẾM và
#   phản hồi lại trong prompt đợt sau. Không model → tiến hoá đột biến thuần.
# ════════════════════════════════════════════════════════════════════════════
import json, math, os, random, re, sys, time, hashlib
from collections import Counter, defaultdict
from pathlib import Path

try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
import torch
import torch.nn.functional as F

U32 = 2.0 ** -24                  # unit roundoff fp32 (round-to-nearest)
ULIBM = 4 * U32                   # cận ulp cho exp/tanh/log (libdevice ≤ 2ulp, x2 dự phòng)
SAFE_MAX = 3.0e38                 # dưới FLT_MAX: chỉ phán xét nơi giá trị thật biểu diễn được
DENORM_SLACK = 2.0 ** -126        # flush-to-zero hợp lệ ở vùng denormal
U16 = 2.0 ** -11                  # unit roundoff fp16 — cho oracle "lý thuyết fp16"
DS16 = 2.0 ** -14                 # min-normal fp16: slack flush-to-zero của half
FP16_SAFE_MAX = 6.0e4             # dưới HALF_MAX=65504: tiền-đề không-tràn cho đường fp16
TAU_GRAY = 0.8
CAP_ISLAND = {"fp32": 2.0 ** 24, "fp16": 2.0 ** 11, "bf16": 2.0 ** 8, "tf32": 2.0 ** 11}
NUMEL_CAP = 1 << 21
SILENT, GRAY, CERT, EXACTV = "SILENT", "GRAY_SUSPECT", "CERTIFIED", "EXACT_VIOLATION"
KNOWN = {"compile_drift", "precision_dtype", "device_divergence", "layout_phase",
         "reassociation", "fused_kernel"}

DEV_CUDA = torch.cuda.is_available()
DEV = "cuda" if DEV_CUDA else "cpu"
CAP_SM = torch.cuda.get_device_capability() if DEV_CUDA else (0, 0)
HAS_TF32 = DEV_CUDA and CAP_SM[0] >= 8

def gamma(k, u=U32):              # γ_k = k·u/(1−k·u), hợp lệ khi k·u < 1
    ku = k * u
    return ku / (1.0 - ku) if ku < 1 else math.inf

def canonical_equal(x, y):        # v2: +0==−0, NaN==NaN — diệt FP signed-zero tận gốc
    if x.shape != y.shape: return False
    x = x.detach().float(); y = y.detach().float()
    return bool(torch.all((x == y) | (torch.isnan(x) & torch.isnan(y)) | ((x == 0) & (y == 0))))

# ════════════════════════════════════════════════════════════════════════════
# DSL: program = {"in":[[r,c],...], "g":[["op",a(,b)],...], "o":out_id}
#   node id: input thứ i là i; node g[j] là len(in)+j. Mọi tensor 2-D.
#   softmax/logsumexp là MACRO → khai triển thành primitive (bao tự hợp thành),
#   executor thêm đường "fused" dùng kernel hợp nhất của torch để đối chiếu.
# ════════════════════════════════════════════════════════════════════════════
UNARY = {"neg", "abs", "relu", "exp", "tanh", "log", "transpose",
         "sum", "sumk", "amax", "amaxk", "cumsum"}
BINARY = {"add", "sub", "mul", "div", "matmul"}
MACROS = {"softmax", "logsumexp"}
ISLAND_OPS = {"neg", "abs", "relu", "transpose", "sum", "sumk", "amax", "amaxk",
              "cumsum", "add", "sub", "mul", "matmul"}
INTERESTING = {"matmul", "exp", "log", "div", "cumsum", "sum", "sumk", "softmax", "logsumexp"}

def _bshape(s1, s2):              # broadcast 2-D kiểu torch; None nếu không hợp
    if len(s1) != 2 or len(s2) != 2: return None
    out = []
    for a, b in zip(s1, s2):
        if a == b or b == 1: out.append(a)
        elif a == 1: out.append(b)
        else: return None
    return tuple(out)

def expand_macros(prog):
    """Khai triển softmax/logsumexp thành primitive; trả (g_mới, spans).
    spans = [(kind, in_id, out_id)] để executor thêm đường fused."""
    n_in = len(prog["in"])
    g, spans, remap = [], [], {}
    def nid(): return n_in + len(g)
    for j, node in enumerate(prog["g"]):
        op, args = node[0], [remap.get(a, a) for a in node[1:]]
        old_id = n_in + j
        if op in MACROS:
            x = args[0]
            g.append(["amaxk", x]); m = nid() - 1
            g.append(["sub", x, m]); t = nid() - 1
            g.append(["exp", t]);    e = nid() - 1
            g.append(["sumk", e]);   s = nid() - 1
            if op == "softmax":
                g.append(["div", e, s]); out = nid() - 1
            else:
                g.append(["log", s]);  l = nid() - 1
                g.append(["add", l, m]); out = nid() - 1
            spans.append((op, x, out)); remap[old_id] = out
        else:
            g.append([op] + args); remap[old_id] = nid() - 1
    return g, spans, remap[n_in + len(prog["g"]) - 1] if prog["g"] else None, remap

def sanitize(raw):
    """Whitelist + suy luận shape TRƯỚC thực thi (diệt lớp invalid-24%-ngân-sách).
    Trả (prog_chuẩn | None, lý_do)."""
    try:
        if not isinstance(raw, dict): return None, "not_dict"
        ins = raw.get("in"); g = raw.get("g")
        if not (isinstance(ins, list) and 1 <= len(ins) <= 4): return None, "bad_inputs"
        if not (isinstance(g, list) and 1 <= len(g) <= 10): return None, "bad_graph_len"
        shapes = []
        for s in ins:
            if not (isinstance(s, list) and len(s) == 2): return None, "input_not_2d"
            r, c = int(s[0]), int(s[1])
            if not (2 <= r <= 1024 and 2 <= c <= 1024 and r * c <= NUMEL_CAP):
                return None, "dim_out_of_range"
            shapes.append((r, c))
        prog = {"in": [list(s) for s in shapes],
                "g": [[str(n[0])] + [int(a) for a in n[1:]] for n in g],
                "o": int(raw.get("o", len(ins) + len(g) - 1))}
        for n in prog["g"]:
            op = n[0]
            if op in UNARY or op in MACROS:
                if len(n) != 2: return None, "arity"
            elif op in BINARY:
                if len(n) != 3: return None, "arity"
            else: return None, "unknown_op:" + op
        gg, spans, out_id, _ = expand_macros(prog)
        if prog["o"] != len(prog["in"]) + len(prog["g"]) - 1:
            return None, "out_not_last"     # quy ước: output = node cuối
        # suy luận shape toàn đồ thị (sau khai triển macro)
        sh = list(shapes)
        for n in gg:
            op, a = n[0], n[1]
            if a >= len(sh) or a < 0: return None, "bad_ref"
            sa = sh[a]
            if op in BINARY:
                b = n[2]
                if b >= len(sh) or b < 0: return None, "bad_ref"
                sb = sh[b]
                if op == "matmul":
                    if sa[1] != sb[0]: return None, "matmul_shape"
                    s = (sa[0], sb[1])
                else:
                    s = _bshape(sa, sb)
                    if s is None: return None, "broadcast"
            elif op == "transpose": s = (sa[1], sa[0])
            elif op in ("sum", "amax"): s = (sa[0], 1)      # giữ 2-D cho đồng nhất
            elif op in ("sumk", "amaxk"): s = (sa[0], 1)
            else: s = sa
            if s[0] * s[1] > NUMEL_CAP: return None, "numel_cap"
            sh.append(s)
        if not (INTERESTING & {n[0] for n in prog["g"]}): return None, "boring"
        ops_flat = [n[0] for n in gg]
        prog["_expanded"] = gg; prog["_spans"] = spans
        prog["_shapes"] = sh; prog["_island"] = all(o in ISLAND_OPS for o in ops_flat)
        prog["_has_mm"] = "matmul" in ops_flat
        prog["_key"] = hashlib.sha1(json.dumps({"in": prog["in"], "g": prog["g"]},
                                               sort_keys=True).encode()).hexdigest()[:10]
        return prog, "ok"
    except Exception as e:
        return None, "exc:" + type(e).__name__

def dim_classes(prog):
    """Union-find các chiều input bị ràng buộc bằng nhau (matmul/broadcast)
    → jitter shape theo LỚP, không bao giờ sinh chương trình lệch shape."""
    n_in = len(prog["in"])
    dims = [(i, d) for i in range(n_in) for d in (0, 1)]
    parent = {x: x for x in dims}
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(x, y):
        if x in parent and y in parent: parent[find(x)] = find(y)
    src = {}                                    # node id -> (nguồn_dim0, nguồn_dim1)
    for i in range(n_in): src[i] = ((i, 0), (i, 1))
    sh = prog["_shapes"]
    for j, n in enumerate(prog["_expanded"]):
        op, a = n[0], n[1]; mid = n_in + j; sa = src.get(a, (None, None))
        if op == "matmul":
            b = n[2]; sb = src.get(b, (None, None))
            if sa[1] and sb[0]: union(sa[1], sb[0])
            src[mid] = (sa[0], sb[1])
        elif op in ("add", "sub", "mul", "div"):
            b = n[2]; sb = src.get(b, (None, None))
            for d in (0, 1):
                if sh[a][d] == sh[b][d] and sh[a][d] > 1 and sa[d] and sb[d]:
                    union(sa[d], sb[d])
            src[mid] = tuple(sa[d] if sh[a][d] >= sh[b][d] else sb[d] for d in (0, 1))
        elif op == "transpose": src[mid] = (sa[1], sa[0])
        elif op in ("sum", "sumk", "amax", "amaxk"): src[mid] = (sa[0], None)
        else: src[mid] = sa
    groups = defaultdict(list)
    for x in dims: groups[find(x)].append(x)
    return [v for v in groups.values()]

# ════════════════════════════════════════════════════════════════════════════
# BỘ NỘI SUY 1: tham chiếu fp64 + BAO CHỨNG MINH E (giải tích sai số thuận).
# Bất biến quy nạp: mọi đánh-giá-fp32-trung-thực x̂ của node có |x̂ − ref| ≤ E.
# E=None nghĩa là 0 (input chính xác) — né matmul fp64 thừa.
# ════════════════════════════════════════════════════════════════════════════
def _Ez(E, ref): return torch.zeros_like(ref) if E is None else E

def envelope_eval(prog, inputs64, u=U32, ds=DENORM_SLACK, safe_max=SAFE_MAX,
                  in_cast_err=0.0):
    """Trả (ref, E, V): tham chiếu fp64, bao chứng minh, và MẶT NẠ HIỆU LỰC V.
    V[i]=True ⇔ mọi giá trị trung gian nuôi vị trí i đều hữu hạn & trong tầm
    biểu diễn & không dính vùng bao=inf (lan truyền "nhiễm" qua matmul/sum/
    cumsum) → chỉ phán xét nơi định lý còn đứng; tràn-hợp-lệ KHÔNG thành báo
    nhầm. Slack denormal ds cộng ở TỪNG op làm tròn: flush-to-zero hợp lệ (vd
    đuôi softmax e^-100 → 0) nằm trong bao, diệt lớp báo nhầm FTZ tận gốc.
    THAM SỐ HOÁ theo độ chính xác: (u=2^-24, ds=2^-126, safe_max~FLT_MAX) là
    hợp đồng fp32; (u=2^-11, ds=2^-14, safe_max~HALF_MAX, in_cast_err=u16) là
    "LÝ THUYẾT fp16 thuần" — đường autocast vượt bao này ⇒ tệ hơn cả mức
    lý thuyết half-precision cho phép (oracle mixed-precision, lớp mới)."""
    DS = ds; ul = 4 * u
    n_in = len(inputs64)
    refs = list(inputs64)
    if in_cast_err > 0:               # sai số cast input (vd fl16(x): ≤ u16|x|, + FTZ)
        Es = [in_cast_err * r.abs() + DS for r in inputs64]
    else:
        Es = [None] * n_in
    Vs = [torch.ones_like(r, dtype=torch.bool) for r in inputs64]
    for n in prog["_expanded"]:
        op, a = n[0], n[1]
        ra, Ea, Va = refs[a], Es[a], Vs[a]
        if op in BINARY:
            b = n[2]; rb, Eb, Vb = refs[b], Es[b], Vs[b]
        if op == "add" or op == "sub":
            ref = ra + rb if op == "add" else ra - rb
            E = (_Ez(Ea, ra) + _Ez(Eb, rb)) * (1 + u) + u * ref.abs() + DS
            V = Va & Vb
        elif op == "mul":
            ref = ra * rb
            Ea_, Eb_ = _Ez(Ea, ra), _Ez(Eb, rb)
            E = (ra.abs() * Eb_ + rb.abs() * Ea_ + Ea_ * Eb_) * (1 + u) + u * ref.abs() + DS
            V = Va & Vb
        elif op == "div":
            ref = ra / rb
            Ea_, Eb_ = _Ez(Ea, ra), _Ez(Eb, rb)
            den = rb.abs() - Eb_
            raw = (Ea_ + ref.abs() * Eb_) / den.clamp_min(1e-300)
            E = torch.where(den > 0, raw * (1 + u) + u * ref.abs() + DS,
                            torch.full_like(ref, math.inf))
            V = Va & Vb
        elif op == "matmul":
            # Tiền-đề W–H: KHÔNG tràn ở tích/tổng-riêng-phần. P chặn trên mọi
            # đại lượng trung gian của dot-product (mọi thứ tự cộng) → P phải
            # trong tầm biểu diễn, nếu không inf−inf=NaN dù kết quả cuối nhỏ.
            k = ra.shape[-1]; g = gamma(k, u)
            ref = ra @ rb
            Aa, Ab = ra.abs(), rb.abs()
            if Ea is None and Eb is None:
                P = Aa @ Ab
                E = g * P + 2 * k * DS
            else:
                Ea_, Eb_ = _Ez(Ea, ra), _Ez(Eb, rb)
                P = (Aa + Ea_) @ (Ab + Eb_)
                E = Ea_ @ Ab + Aa @ Eb_ + Ea_ @ Eb_ + g * P + 2 * k * DS
            V = Va.all(-1, keepdim=True) & Vb.all(-2, keepdim=True) & (P < safe_max)
        elif op == "neg":   ref, E, V = -ra, Ea, Va
        elif op == "abs":   ref, E, V = ra.abs(), Ea, Va
        elif op == "relu":  ref, E, V = ra.relu(), Ea, Va             # 1-Lipschitz, op chính xác
        elif op == "tanh":
            ref = torch.tanh(ra); E = _Ez(Ea, ra) + ul + DS; V = Va
        elif op == "exp":
            ref = torch.exp(ra)
            Ea_ = _Ez(Ea, ra)
            E = ref * torch.expm1(Ea_) + ul * ref * torch.exp(Ea_) + DS
            V = Va
        elif op == "log":
            ref = torch.log(ra)
            Ea_ = _Ez(Ea, ra); den = ra - Ea_
            E = torch.where(den > 0, Ea_ / den.clamp_min(1e-300) + ul * ref.abs() + u + DS,
                            torch.full_like(ra, math.inf))
            V = Va
        elif op == "transpose":
            ref, E, V = ra.mT, (None if Ea is None else Ea.mT), Va.mT
        elif op in ("sum", "sumk"):
            kk = ra.shape[-1]; g = gamma(max(kk - 1, 1), u)
            ref = ra.sum(-1, keepdim=True)
            Ea_ = _Ez(Ea, ra)
            S = (ra.abs() + Ea_).sum(-1, keepdim=True)       # chặn tổng-riêng-phần
            E = Ea_.sum(-1, keepdim=True) + g * S + kk * DS
            V = Va.all(-1, keepdim=True) & (S < safe_max)
        elif op in ("amax", "amaxk"):
            ref = ra.amax(-1, keepdim=True)
            E = None if Ea is None else Ea.amax(-1, keepdim=True)
            V = Va.all(-1, keepdim=True)
        elif op == "cumsum":
            kk = ra.shape[-1]; g = gamma(max(kk - 1, 1), u)
            ref = ra.cumsum(-1)
            Ea_ = _Ez(Ea, ra)
            S = (ra.abs() + Ea_).cumsum(-1)                  # chặn tổng-riêng-phần
            E = Ea_.cumsum(-1) + g * S + kk * DS
            V = (torch.cumprod(Va.to(torch.float64), -1) > 0.5) & (S < safe_max)
        else:
            raise RuntimeError("op? " + op)
        m = ref.abs() + (0 if E is None else E)
        V = (V & (m < safe_max)).broadcast_to(ref.shape)              # NaN/Inf tự rớt
        refs.append(ref); Es.append(E); Vs.append(V)
    out = len(refs) - 1
    return refs[out], _Ez(Es[out], refs[out]), Vs[out]

# ════════════════════════════════════════════════════════════════════════════
# BỘ NỘI SUY 2: cận biên độ M cho ĐẢO-CHÍNH-XÁC (giá trị nguyên ⇒ exact).
# ════════════════════════════════════════════════════════════════════════════
def magnitude_max(prog, rngs):
    Ms = [float(r) for r in rngs]; peak = max(Ms)
    sh = prog["_shapes"]; n_in = len(prog["in"])
    for j, n in enumerate(prog["_expanded"]):
        op, a = n[0], n[1]; Ma = Ms[a]
        if op in ("add", "sub"): M = Ma + Ms[n[2]]
        elif op == "mul":        M = Ma * Ms[n[2]]
        elif op == "matmul":     M = sh[a][1] * Ma * Ms[n[2]]
        elif op in ("sum", "sumk", "cumsum"): M = sh[a][1] * Ma
        else:                    M = Ma           # neg/abs/relu/amax*/transpose
        Ms.append(M); peak = max(peak, M)
    return peak

# ════════════════════════════════════════════════════════════════════════════
# EXECUTOR: chạy đồ thị fp32 trên các đường bảo-toàn-ngữ-nghĩa.
# ════════════════════════════════════════════════════════════════════════════
def _apply(op, vals, a, b):
    x = vals[a]
    if op == "add": return x + vals[b]
    if op == "sub": return x - vals[b]
    if op == "mul": return x * vals[b]
    if op == "div": return x / vals[b]
    if op == "matmul": return x @ vals[b]
    if op == "neg": return -x
    if op == "abs": return x.abs()
    if op == "relu": return x.relu()
    if op == "exp": return torch.exp(x)
    if op == "tanh": return torch.tanh(x)
    if op == "log": return torch.log(x)
    if op == "transpose": return x.mT
    if op == "sum": return x.sum(-1, keepdim=True)
    if op == "sumk": return x.sum(-1, keepdim=True)
    if op == "amax": return x.amax(-1, keepdim=True)
    if op == "amaxk": return x.amax(-1, keepdim=True)
    if op == "cumsum": return x.cumsum(-1)
    raise RuntimeError(op)

def make_runner(expanded, n_in, splitk=False, fused_spans=None):
    fused = {out: (kind, src) for kind, src, out in (fused_spans or [])}
    def run(*xs):
        vals = list(xs)
        for n in expanded:
            op = n[0]; a = n[1]; b = n[2] if len(n) > 2 else 0
            if op == "matmul" and splitk and vals[a].shape[-1] >= 4:
                A, B = vals[a], vals[b]; h = A.shape[-1] // 2
                y = A[..., :h] @ B[..., :h, :] + A[..., h:] @ B[..., h:, :]
            else:
                y = _apply(op, vals, a, b)
            i = len(vals)
            if i in fused:
                kind, src = fused[i]
                y = (F.softmax(vals[src], dim=-1) if kind == "softmax"
                     else torch.logsumexp(vals[src], dim=-1, keepdim=True))
            vals.append(y)
        return vals[-1]
    return run

_COMPILE_CACHE, _COMPILE_FAILED = {}, [False]
def compiled_runner(prog, budget):
    key = prog["_key"]
    if key in _COMPILE_CACHE: return _COMPILE_CACHE[key]
    if _COMPILE_FAILED[0] or budget[0] <= 0: return None
    try:
        fn = torch.compile(make_runner(prog["_expanded"], len(prog["in"])), dynamic=True)
        _COMPILE_CACHE[key] = fn; budget[0] -= 1
        return fn
    except Exception as e:
        _COMPILE_FAILED[0] = True
        print(">> torch.compile không khả dụng (%s) — bỏ đường compiled." % type(e).__name__)
        return None

def run_paths(prog, inputs, mode, peakM, compile_budget):
    """Trả dict path -> tensor fp32 (trên DEV). inputs: list fp32 trên DEV."""
    n_in = len(inputs)
    base = make_runner(prog["_expanded"], n_in)
    paths = {"eager": base(*inputs)}
    cfn = compiled_runner(prog, compile_budget)
    if cfn is not None:
        try: paths["compiled"] = cfn(*inputs)
        except Exception: pass
    if DEV == "cuda":
        try:
            cpu_in = [x.cpu() for x in inputs]
            paths["vs_cpu"] = base(*cpu_in).to(DEV)
        except Exception: pass
    try:
        lay = [x.mT.contiguous().mT for x in inputs]      # cùng giá trị, khác stride
        paths["layout"] = base(*lay)
    except Exception: pass
    if prog["_has_mm"]:
        try: paths["splitk"] = make_runner(prog["_expanded"], n_in, splitk=True)(*inputs)
        except Exception: pass
    if prog["_spans"]:
        try: paths["fused"] = make_runner(prog["_expanded"], n_in,
                                          fused_spans=prog["_spans"])(*inputs)
        except Exception: pass
    if mode == "float" and HAS_TF32 and prog["_has_mm"]:
        old = torch.backends.cuda.matmul.allow_tf32
        try:
            torch.backends.cuda.matmul.allow_tf32 = True
            paths["tf32"] = base(*inputs).clone()
        finally:
            torch.backends.cuda.matmul.allow_tf32 = old
    if mode == "island" and DEV == "cuda":
        for dt, nm, cap in [(torch.float16, "amp16", CAP_ISLAND["fp16"]),
                            (torch.bfloat16, "ampb16", CAP_ISLAND["bf16"])]:
            if peakM < cap:
                try:
                    with torch.autocast(device_type="cuda", dtype=dt):
                        paths[nm] = base(*inputs)
                    paths[nm] = paths[nm].float()
                except Exception: pass
        if HAS_TF32 and prog["_has_mm"] and peakM < CAP_ISLAND["tf32"]:
            old = torch.backends.cuda.matmul.allow_tf32
            try:
                torch.backends.cuda.matmul.allow_tf32 = True
                paths["tf32"] = base(*inputs).clone()
            finally:
                torch.backends.cuda.matmul.allow_tf32 = old
    return paths

def path_class(path):
    return {"compiled": "compile_drift", "vs_cpu": "device_divergence",
            "layout": "layout_phase", "splitk": "reassociation",
            "fused": "fused_kernel", "tf32": "precision_dtype",
            "amp16": "precision_dtype", "ampb16": "precision_dtype",
            "amp16_theory": "amp_beyond_fp16_theory"}.get(path, "op_core")

# ════════════════════════════════════════════════════════════════════════════
# GRADER
# ════════════════════════════════════════════════════════════════════════════
def grade_island(prog, inputs, peakM, compile_budget):
    """Mọi đường phải khớp BIT với kết quả nguyên chính xác. Trả list finding-row."""
    ins64 = [x.double() for x in inputs]
    ref = make_runner(prog["_expanded"], len(inputs))(*ins64).float()   # exact: M<2^24
    rows = []
    for nm, t in run_paths(prog, inputs, "island", peakM, compile_budget).items():
        ok = canonical_equal(t, ref)
        if not ok:
            d = (t.double() - ref.double()).abs()
            rows.append((nm, EXACTV, math.inf,
                         "exact island broken: max|Δ|=%.3g tại %d/%d phần tử"
                         % (d.max().item(), int((d > 0).sum()), d.numel())))
        else:
            rows.append((nm, SILENT, 0.0, "bit-exact"))
    return rows

def _grade_one(nm, o, ref, E, V, floor):
    if o.shape != ref.shape:
        return (nm, CERT, math.inf, "shape mismatch %s vs %s"
                % (tuple(o.shape), tuple(ref.shape)))
    if ((~torch.isfinite(o)) & V).any():
        return (nm, CERT, math.inf, "non-finite tại vị trí chứng-minh-được hữu hạn")
    d = torch.where(V, (o - ref).abs(), torch.zeros_like(ref))
    ratio = float((d / E.clamp_min(floor)).max())
    lab = CERT if ratio > 1.0 else (GRAY if ratio > TAU_GRAY else SILENT)
    return (nm, lab, ratio, "cert_ratio=%.3f" % ratio)

def grade_float(prog, inputs, compile_budget, amp16=False):
    ins64 = [x.double() for x in inputs]
    ref, E, V = envelope_eval(prog, ins64)
    if not V.any():
        return None                       # không còn vị trí chứng-minh-được — bỏ instance
    rows = []
    for nm, t in run_paths(prog, inputs, "float", 0, compile_budget).items():
        rows.append(_grade_one(nm, t.double(), ref, E, V, DENORM_SLACK))
    if amp16 and DEV == "cuda":
        # ORACLE MIXED-PRECISION: chạy cả chương trình dưới autocast fp16, phán
        # theo bao "lý thuyết fp16 thuần" (u=2^-11, FTZ 2^-14, HALF_MAX). Vượt
        # bao ⇒ autocast tệ hơn cả mức lý thuyết half cho phép — lớp MỚI,
        # không nằm trong KNOWN. (autocast giữ elementwise ở fp32 nên bao
        # all-u16 là nới-an-toàn: chỉ thiếu nhạy, không thể kêu oan.)
        try:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                o16 = make_runner(prog["_expanded"], len(inputs))(*inputs)
            ref2, E16, V16 = envelope_eval(prog, ins64, u=U16, ds=DS16,
                                           safe_max=FP16_SAFE_MAX, in_cast_err=U16)
            if V16.any():
                rows.append(_grade_one("amp16_theory", o16.float().double(),
                                       ref2, E16, V16, DS16))
        except Exception:
            pass
    return rows

# ════════════════════════════════════════════════════════════════════════════
# SINH GIÁ TRỊ + INSTANCE
# ════════════════════════════════════════════════════════════════════════════
FLOAT_STYLES = ["normal", "cancel", "bigscale", "rowscale"]
def gen_inputs(prog, mode, style, seed):
    torch.manual_seed(seed)
    outs = []
    for (r, c) in (tuple(s) for s in prog["in"]):
        if mode == "island":
            rng = style                                   # style = bán kính nguyên
            x = torch.randint(-rng, rng + 1, (r, c), device=DEV).float()
        else:
            x = torch.randn(r, c, device=DEV) * 0.5
            if style == "cancel" and c % 2 == 0:
                x[:, 1::2] = -x[:, 0::2]                  # bài học 5: triệt tiêu đối kháng
            elif style == "bigscale":
                x = x * 256.0
            elif style == "rowscale":
                x = x * torch.pow(2.0, torch.randint(-8, 9, (r, 1), device=DEV).float())
        outs.append(x)
    return outs

def island_radius(prog):
    for rng in (2, 1):
        if magnitude_max(prog, [rng] * len(prog["in"])) < CAP_ISLAND["fp32"]:
            return rng
    return None

def jitter(prog, classes, rng):
    """Đột biến kích thước theo LỚP chiều ràng buộc → luôn hợp shape."""
    new_in = [list(s) for s in prog["in"]]
    cls = rng.choice(classes)
    cur = new_in[cls[0][0]][cls[0][1]]
    nv = max(2, min(1024, cur + rng.choice([-17, -16, -1, 1, 15, 16, 31, 32, 64])))
    for (i, d) in cls: new_in[i][d] = nv
    cand = {"in": new_in, "g": [list(n) for n in prog["g"]],
            "o": len(new_in) + len(prog["g"]) - 1}
    p2, why = sanitize(cand)
    return p2

def mutate(prog, rng):
    """Đột biến cấu trúc: chèn unary vào mép ngẫu nhiên."""
    g = [list(n) for n in prog["g"]]
    if len(g) < 10 and rng.random() < 0.7:
        j = rng.randrange(len(g))
        tgt = len(prog["in"]) + j
        ins = [rng.choice(["relu", "abs", "neg", "tanh", "cumsum", "exp"]), tgt]
        g2 = g[:j + 1] + [ins] + [[n[0]] + [a + 1 if a > tgt else a for a in n[1:]]
                                  for n in g[j + 1:]]
        # các node sau trỏ tới tgt giờ trỏ node mới (nuôi chuỗi sâu hơn)
        for n in g2[j + 2:]:
            for ii in range(1, len(n)):
                if n[ii] == tgt: n[ii] = tgt + 1
        cand = {"in": [list(s) for s in prog["in"]], "g": g2,
                "o": len(prog["in"]) + len(g2) - 1}
        p2, _ = sanitize(cand)
        if p2 is not None: return p2
    cls = dim_classes(prog)
    return jitter(prog, cls, rng) or prog

# ════════════════════════════════════════════════════════════════════════════
# THƯ VIỆN HẠT GIỐNG (qua cùng sanitizer — ăn cơm nhà mình nấu)
# ════════════════════════════════════════════════════════════════════════════
SEED_RAW = [
    {"in": [[64, 96], [96, 48], [48, 32]],
     "g": [["matmul", 0, 1], ["relu", 3], ["matmul", 4, 2]], "o": 5},
    {"in": [[32, 128]], "g": [["cumsum", 0], ["neg", 0], ["cumsum", 2],
                              ["neg", 3], ["sub", 1, 4]], "o": 5},
    {"in": [[48, 256]], "g": [["softmax", 0]], "o": 1},
    {"in": [[40, 512]], "g": [["logsumexp", 0]], "o": 1},
    {"in": [[96, 64], [64, 96]], "g": [["matmul", 0, 1], ["cumsum", 2], ["sum", 3]], "o": 4},
    {"in": [[64, 64], [64, 64]], "g": [["mul", 0, 1], ["exp", 2], ["sum", 3]], "o": 4},
    {"in": [[128, 32], [32, 128]], "g": [["abs", 0], ["matmul", 2, 1], ["amax", 3]], "o": 4},
    {"in": [[80, 80], [80, 80]], "g": [["transpose", 0], ["matmul", 2, 1],
                                       ["neg", 3], ["relu", 4]], "o": 5},
    {"in": [[64, 128]], "g": [["amaxk", 0], ["sub", 0, 1], ["exp", 2],
                              ["sumk", 3], ["div", 3, 4]], "o": 5},
    {"in": [[56, 192], [192, 56]], "g": [["matmul", 0, 1], ["tanh", 2],
                                         ["matmul", 3, 0]], "o": 4},
]

# ════════════════════════════════════════════════════════════════════════════
# CHÍNH SÁCH GEMMA 4: sinh CHƯƠNG TRÌNH theo lô. Máy chứng minh gác cổng —
# model dở chỉ phí ngân sách, không tạo nổi bug giả.
# ════════════════════════════════════════════════════════════════════════════
GRAMMAR = ('{"in":[[r,c],...](2-D, 2..1024), "g":[["op",a(,b)],...], "o":last}. '
           'ids: inputs 0..n-1, node j = n+j; "o" MUST be the last node id. unary: '
           "neg,abs,relu,exp,tanh,log,transpose,sum,amax,cumsum,softmax,logsumexp; "
           "binary: add,sub,mul,div,matmul(2D: [a,k]x[k,b]). <=10 nodes.")
FEWSHOT = ('{"in":[[64,96],[96,32]],"g":[["matmul",0,1],["relu",2],["cumsum",3]],"o":4}\n'
           '{"in":[[48,256]],"g":[["softmax",0]],"o":1}')

class MutationPolicy:
    name = "mutation"
    def __init__(self, seeds, rng): self.seeds, self.rng = seeds, rng
    def generate(self, summary, n=6):
        outs = []
        for _ in range(n):
            p = mutate(self.rng.choice(self.seeds), self.rng)
            if p is not None: outs.append(p)
        return outs, {"parse_fail": 0, "asked": n}

class GemmaProgramPolicy:
    """Một lần gọi → NHIỀU chương trình (sampling, num_return_sequences). Prompt
    mang phản hồi đợt trước (cell nóng, họ-op đã im, số lỗi cú pháp) → model tự
    sửa. Nhận (model, tok) TIÊM TỪ NGOÀI (notebook đã load sẵn Gemma 4) hoặc tự
    load qua env GEMMA_DIR/GEMMA_ID khi chạy standalone."""
    name = "gemma4-program"
    SYS = ("You write short tensor programs for a CERTIFIED numerical-reliability "
           "tester of tensor libraries (scientific-software QA, TitanFuzz/NNSmith line). "
           "The prover checks each program on equivalent execution paths (compiled "
           "fusion, device, layout, reassociation, fused kernels, autocast) against "
           "proven error envelopes; you cannot cause false alarms, only focus coverage. "
           "Favor structures that stress fusion and accumulation: matmul chains, "
           "cumsum/sum after matmul, softmax/logsumexp, cancellation-friendly mixes. "
           "DSL: " + GRAMMAR +
           "\nReply with EXACTLY one JSON program per line, no prose, no markdown. "
           "Examples:\n" + FEWSHOT)
    def __init__(self, model, tok, fallback=None):
        self.model, self.tok, self.fallback = model, tok, fallback
        self.last_fail = 0
        if getattr(tok, "pad_token", None) is None and getattr(tok, "eos_token", None):
            tok.pad_token = tok.eos_token
        if hasattr(tok, "padding_side"): tok.padding_side = "left"

    @classmethod
    def load(cls, fallback):
        src = os.environ.get("GEMMA_DIR") or os.environ.get("GEMMA_ID")
        if not src: return None
        try:
            from transformers import AutoTokenizer
            kw = {"dtype": "auto", "device_map": "auto", "low_cpu_mem_usage": True}
            if os.environ.get("GEMMA_4BIT", "1") == "1" and DEV_CUDA:
                from transformers import BitsAndBytesConfig
                cdt = torch.float16 if CAP_SM[0] < 8 else torch.bfloat16   # T4: fp16
                kw["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True, bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=cdt, bnb_4bit_use_double_quant=True)
            local = os.path.isdir(src)
            tok = AutoTokenizer.from_pretrained(src, local_files_only=local)
            model = None; last = None
            import transformers as TF
            for cn in ("AutoModelForMultimodalLM", "AutoModelForImageTextToText",
                       "AutoModelForCausalLM"):           # chuỗi loader đã chứng minh trên T4
                c = getattr(TF, cn, None)
                if c is None: continue
                try:
                    model = c.from_pretrained(src, local_files_only=local, **kw); break
                except Exception as e: last = e
            if model is None: raise last
            model.eval()
            print(">> GemmaProgramPolicy loaded:", src)
            return cls(model, tok, fallback)
        except Exception as e:
            print(">> Gemma load failed → mutation policy:", str(e)[:100])
            return None

    def _chat(self, user):
        msgs = [{"role": "system", "content": self.SYS},
                {"role": "user", "content": user}]
        merged = [{"role": "user", "content": self.SYS + "\n\n" + user}]
        tmpl = getattr(self.tok, "apply_chat_template", None)
        if tmpl:
            for mm, kw in ((msgs, {"enable_thinking": False}), (msgs, {}),
                           (merged, {"enable_thinking": False}), (merged, {})):
                try:
                    return tmpl(mm, tokenize=False, add_generation_prompt=True, **kw)
                except Exception:
                    continue
        return self.SYS + "\n" + user + "\nJSON:"

    @staticmethod
    def _scan_json(text):
        """Quét mọi object JSON trong text (kể cả kẹp markdown) bằng raw_decode."""
        text = text.replace("```json", " ").replace("```", " ")
        dec, out, i = json.JSONDecoder(), [], 0
        while True:
            j = text.find("{", i)
            if j < 0: break
            try:
                obj, end = dec.raw_decode(text[j:])
                out.append(obj); i = j + end
            except Exception:
                i = j + 1
        return out

    def generate(self, summary, n=6):
        try:
            user = ("coverage summary: %s\nlast batch: %d JSON/grammar errors — obey the "
                    "format strictly. Emit %d NEW diverse programs (op mixes different "
                    "from hot cells, 3-8 nodes, include matmul or softmax/logsumexp or "
                    "cumsum), one JSON per line."
                    % (json.dumps(summary)[:700], self.last_fail, n))
            enc = self.tok(self._chat(user), return_tensors="pt",
                           truncation=True, max_length=1024).to(self.model.device)
            kw = dict(max_new_tokens=384, do_sample=True, temperature=0.9,
                      top_p=0.95, top_k=64, num_return_sequences=2)
            if getattr(self.tok, "pad_token_id", None) is not None:
                kw["pad_token_id"] = self.tok.pad_token_id
            with torch.inference_mode():
                gen = self.model.generate(**enc, **kw)
            progs, fails = [], 0
            for row in gen:
                txt = self.tok.decode(row[enc["input_ids"].shape[1]:],
                                      skip_special_tokens=True)
                for raw in self._scan_json(txt):
                    p, why = sanitize(raw)
                    if p is not None: progs.append(p)
                    else: fails += 1
            self.last_fail = fails
            if not progs:                                  # an toàn tuyệt đối
                out, st = self.fallback.generate(summary, n)
                st["parse_fail"] = fails or 1
                return out, st
            return progs[: n * 2], {"parse_fail": fails, "asked": n}
        except Exception as e:
            print(">> Gemma generate lỗi → fallback:", str(e)[:80])
            return self.fallback.generate(summary, n)

# ════════════════════════════════════════════════════════════════════════════
# GRADIENT POLISH (fuzzing khả vi): leo cert_ratio theo GIÁ TRỊ input trên
# đường eager — genome liên tục; genome rời rạc đã có MAP-Elites + Gemma.
# ════════════════════════════════════════════════════════════════════════════
def grad_polish(prog, inputs, steps=5):
    try:
        xs = [x.clone().requires_grad_(True) for x in inputs]
        opt = torch.optim.Adam(xs, lr=0.05)
        run = make_runner(prog["_expanded"], len(xs))
        for _ in range(steps):
            opt.zero_grad()
            ref, E, V = envelope_eval(prog, [x.double() for x in xs])
            obs = run(*xs).double()
            ok = V & torch.isfinite(obs)
            r = torch.where(ok, (obs - ref).abs() / E.clamp_min(DENORM_SLACK),
                            torch.zeros_like(ref))
            (-(r.max())).backward()
            opt.step()
        return [x.detach() for x in xs]
    except Exception:
        return inputs

# ════════════════════════════════════════════════════════════════════════════
# ENGINE: MAP-Elites trên descriptor chương trình + nhân instance + chứng chỉ.
# ════════════════════════════════════════════════════════════════════════════
def descriptor(prog, mode, path):
    sig = ",".join(sorted(set(n[0] for n in prog["g"])))
    return (sig, min(len(prog["g"]), 8), mode, path)

class EngineV4:
    def __init__(self, seed=0):
        self.rng = random.Random(seed)
        self.archive = {}            # descriptor -> (sev, prog, inst)
        self.findings = {}
        self.programs = {}           # key -> prog (dedup)
        self.checks = 0; self.skips = 0; self.invalid = 0
        self.counts = Counter()
        self.compile_budget = [int(os.environ.get("COMPILE_BUDGET", "24"))]
        if os.environ.get("COMPILE", "1") == "0": _COMPILE_FAILED[0] = True
        self.seeds = []
        for raw in SEED_RAW:
            p, why = sanitize(raw)
            assert p is not None, "seed hỏng: %s (%s)" % (raw, why)
            self.seeds.append(p); self.programs[p["_key"]] = p

    # ---- chấm 1 instance (prog, mode, style, seed) trên mọi đường ----------
    def evaluate(self, prog, mode, style, vseed, shrink=True):
        try:
            inputs = gen_inputs(prog, mode, style, vseed)
            if mode == "island":
                peakM = magnitude_max(prog, [style] * len(prog["in"]))
                rows = grade_island(prog, inputs, peakM, self.compile_budget)
            else:
                amp = DEV_CUDA and prog["_has_mm"] and self.rng.random() < 0.5
                rows = grade_float(prog, inputs, self.compile_budget, amp16=amp)
            if rows is None:
                self.skips += 1; return 0.0
        except Exception:
            self.invalid += 1; return 0.0
        best = 0.0
        for path, lab, sev, detail in rows:
            self.checks += 1; self.counts[lab] += 1
            d = descriptor(prog, mode, path)
            cur = self.archive.get(d)
            if cur is None or sev > cur[0]:
                self.archive[d] = (sev, prog, (mode, style, vseed))
            best = max(best, 0.0 if sev == math.inf else min(sev, 1e6))
            if lab in (CERT, EXACTV) or (lab == GRAY and sev > 0.92):
                fk = (prog["_key"], mode, path, lab)
                prev = self.findings.get(fk)
                sev_n = None if sev == math.inf else round(sev, 4)
                if prev is None or (sev_n or 1e18) > (prev.get("severity") or 0):
                    fnd = {"program": {"in": prog["in"], "g": prog["g"], "o": prog["o"]},
                           "mode": mode, "style": style, "value_seed": vseed,
                           "path": path, "label": lab, "severity": sev_n,
                           "detail": detail, "known_class": path_class(path),
                           "shapes_in": prog["in"]}
                    if shrink and lab in (CERT, EXACTV):
                        fnd["shrunk_in"] = self.shrink(prog, mode, style, vseed, path)
                    self.findings[fk] = fnd
        return best

    def shrink(self, prog, mode, style, vseed, path):
        """Thu nhỏ tự động: halving theo lớp chiều, giữ vi phạm sống."""
        cur = prog
        for _ in range(8):
            classes = dim_classes(cur); moved = False
            for cls in classes:
                v = cur["in"][cls[0][0]][cls[0][1]]
                if v <= 4: continue
                new_in = [list(s) for s in cur["in"]]
                for (i, d) in cls: new_in[i][d] = max(2, v // 2)
                cand, _ = sanitize({"in": new_in, "g": [list(n) for n in cur["g"]],
                                    "o": len(new_in) + len(cur["g"]) - 1})
                if cand is None: continue
                try:
                    inputs = gen_inputs(cand, mode, style, vseed)
                    if mode == "island":
                        pm = magnitude_max(cand, [style] * len(cand["in"]))
                        rows = grade_island(cand, inputs, pm, self.compile_budget)
                    else:
                        rows = grade_float(cand, inputs, self.compile_budget,
                                           amp16=(path == "amp16_theory")) or []
                except Exception:
                    continue
                hit = any(p == path and l in (CERT, EXACTV) for p, l, s, _ in rows)
                if hit: cur = cand; moved = True; break
            if not moved: break
        return cur["in"]

    # ---- nhân 1 chương trình ra nhiều instance --------------------------------
    def sweep(self, prog, n_jitter=2):
        self.programs.setdefault(prog["_key"], prog)
        variants = [prog]
        classes = dim_classes(prog)
        for _ in range(n_jitter):
            v = jitter(prog, classes, self.rng)
            if v is not None: variants.append(v)
        for v in variants:
            r = island_radius(v) if v["_island"] else None
            if r is not None:
                for s in {r, 1}:
                    self.evaluate(v, "island", s, self.rng.randrange(10 ** 6))
            for st in self.rng.sample(FLOAT_STYLES, 2):
                self.evaluate(v, "float", st, self.rng.randrange(10 ** 6))

    def summary(self):
        hot = sorted(((d, v[0]) for d, v in self.archive.items()),
                     key=lambda kv: -kv[1])[:4]
        silent_sigs = sorted({d[0] for d, v in self.archive.items() if v[0] == 0})[:6]
        return {"cells": len(self.archive), "findings": len(self.findings),
                "hot": ["%s|%s|%s sev=%.2f" % (d[0][:28], d[2], d[3], s) for d, s in hot],
                "all_silent_families": silent_sigs}

    def run(self, minutes, policy, burst_sec=180, polish=True):
        t0 = time.time(); deadline = t0 + minutes * 60
        for p in self.seeds: self.sweep(p, n_jitter=1)
        next_burst = time.time() + (5 if policy.name != "mutation" else burst_sec)
        next_log = time.time() + 15; bursts = 0
        while time.time() < deadline:
            if time.time() >= next_burst:
                tb = time.time()
                progs, stats = policy.generate(self.summary(), n=6)
                for p in progs: self.sweep(p, n_jitter=2)
                bursts += 1
                print("  ~ burst #%d (%s): +%d chương trình, %d lỗi cú pháp, %.1fs"
                      % (bursts, policy.name, len(progs), stats.get("parse_fail", 0),
                         time.time() - tb))
                if polish and DEV_CUDA:
                    fl = [(s, pr, ins) for (d, (s, pr, ins)) in self.archive.items()
                          if d[2] == "float" and 0 < s < 1.0]
                    if fl:
                        s, pr, (mode, st, vs) = max(fl, key=lambda r: r[0])
                        xs = grad_polish(pr, gen_inputs(pr, mode, st, vs))
                        try:
                            rows = grade_float(pr, xs, self.compile_budget) or []
                            for path, lab, sev, det in rows:
                                self.checks += 1; self.counts[lab] += 1
                                if lab in (CERT, GRAY):
                                    d2 = descriptor(pr, "float", path)
                                    if sev > self.archive.get(d2, (0,))[0]:
                                        self.archive[d2] = (sev, pr, (mode, st, vs))
                        except Exception: pass
                if DEV_CUDA: torch.cuda.empty_cache()      # trả VRAM sau đợt generate
                next_burst = time.time() + burst_sec
            # tiến hoá nền: tournament từ archive / seeds
            if self.archive and self.rng.random() < 0.7:
                cand = [self.rng.choice(list(self.archive.values())) for _ in range(2)]
                parent = max(cand, key=lambda v: v[0])[1]
            else:
                parent = self.rng.choice(self.seeds)
            child = mutate(parent, self.rng)
            if child is None: continue
            self.sweep(child, n_jitter=1)
            if time.time() >= next_log:
                el = time.time() - t0
                print("  [%6d check] %5.0f check/s | prog %3d | cells %3d | %s | skip %d inv %d | %.0fs"
                      % (self.checks, self.checks / max(el, 1e-9), len(self.programs),
                         len(self.archive),
                         " ".join("%s %d" % (k[:4].lower(), v) for k, v in
                                  sorted(self.counts.items())),
                         self.skips, self.invalid, el))
                next_log = time.time() + 15
        return self

# ════════════════════════════════════════════════════════════════════════════
# PHASE 0 — TỰ KIỂM MÁY CHỨNG MINH (hỏng thì DỪNG, không fuzz bằng oracle gãy)
# ════════════════════════════════════════════════════════════════════════════
def phase0():
    print("-" * 80)
    print("PHASE 0: tự kiểm máy chứng minh (CPU, nhanh)")
    assert gamma(64) > 64 * U32 and gamma(2) < 1e-6
    eng = EngineV4(seed=123)
    rng = random.Random(7)
    # (a) đảo-chính-xác: eager CPU phải khớp bit fp64-cast trên 20 chương trình
    bad = 0
    for p in eng.seeds + [mutate(rng.choice(eng.seeds), rng) for _ in range(10)]:
        if p is None or not p["_island"]: continue
        r = island_radius(p)
        if r is None: continue
        ins = gen_inputs(p, "island", r, rng.randrange(10 ** 6))
        ins_cpu = [x.cpu() for x in ins]
        ref = make_runner(p["_expanded"], len(ins))(*[x.double() for x in ins_cpu]).float()
        got = make_runner(p["_expanded"], len(ins))(*ins_cpu)
        if not canonical_equal(got, ref): bad += 1
    print("  (a) island bit-exact CPU: %s" % ("PASS" if bad == 0 else "FAIL x%d" % bad))
    # (b) bao chứng minh: 40 instance float ngẫu nhiên, eager CPU PHẢI ≤ 1
    worst = 0.0; n_ok = 0
    for _ in range(40):
        p = mutate(rng.choice(eng.seeds), rng)
        if p is None: continue
        ins = [x.cpu() for x in gen_inputs(p, "float", rng.choice(FLOAT_STYLES),
                                           rng.randrange(10 ** 6))]
        try:
            ref, E, V = envelope_eval(p, [x.double() for x in ins])
            if not V.any(): continue
            obs = make_runner(p["_expanded"], len(ins))(*ins).double()
            d = torch.where(V, (obs - ref).abs(), torch.zeros_like(ref))
            rr = float((d / E.clamp_min(DENORM_SLACK)).max())
            if ((~torch.isfinite(obs)) & V).any(): rr = math.inf
            worst = max(worst, rr); n_ok += 1
        except Exception:
            continue
    stat = "PASS" if worst <= 1.0 else "FAIL (định lý vỡ hoặc bao sai!)"
    print("  (b) eager-trong-bao trên %d instance: worst cert_ratio=%.4f → %s"
          % (n_ok, worst, stat))
    # (c) sanitizer chặn rác
    junk = [{"in": [[8, 8]], "g": [["matmul", 0, 0]], "o": 1},          # 8x8@8x8 ok thật ra
            {"in": [[8, 9]], "g": [["matmul", 0, 0]], "o": 1},          # lệch shape
            {"in": [[8, 8]], "g": [["rm -rf", 0]], "o": 1},
            {"in": [[99999, 2]], "g": [["relu", 0]], "o": 1},
            "không phải dict"]
    rej = sum(1 for r in junk if sanitize(r)[0] is None)
    print("  (c) sanitizer: chặn %d/%d mẫu hỏng (mẫu đầu hợp lệ thật)" % (rej, len(junk)))
    # (d) trên CUDA: bao "lý thuyết fp16" phải BAO được autocast của torch lành
    worst16 = 0.0; n16 = 0
    if DEV_CUDA:
        for _ in range(12):
            p = mutate(rng.choice(eng.seeds), rng)
            if p is None or not p["_has_mm"]: continue
            ins = gen_inputs(p, "float", "normal", rng.randrange(10 ** 6))
            try:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    o16 = make_runner(p["_expanded"], len(ins))(*ins)
                ref, E16, V16 = envelope_eval(p, [x.double() for x in ins], u=U16,
                                              ds=DS16, safe_max=FP16_SAFE_MAX,
                                              in_cast_err=U16)
                if not V16.any(): continue
                o = o16.float().double()
                d = torch.where(V16, (o - ref).abs(), torch.zeros_like(ref))
                rr = float((d / E16.clamp_min(DS16)).max())
                if ((~torch.isfinite(o)) & V16).any(): rr = math.inf
                worst16 = max(worst16, rr); n16 += 1
            except Exception:
                continue
        print("  (d) autocast-trong-bao-fp16 trên %d instance: worst=%.4f → %s"
              % (n16, worst16, "PASS" if worst16 <= 1.0 else "FAIL"))
    ok = bad == 0 and worst <= 1.0 and rej >= 4 and worst16 <= 1.0
    print("PHASE 0:", "PASS — máy chứng minh lành" if ok else "FAIL — DỪNG")
    if not ok: sys.exit(2)

# ════════════════════════════════════════════════════════════════════════════
def do_repro(path, idx):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    f = data["findings"][idx]
    p, why = sanitize(f["program"])
    assert p is not None, why
    eng = EngineV4()
    print("REPRO finding #%d: %s/%s (%s) shapes=%s" % (idx, f["mode"], f["path"],
                                                       f["label"], f["shapes_in"]))
    inputs = gen_inputs(p, f["mode"], f["style"], f["value_seed"])
    rows = (grade_island(p, inputs, magnitude_max(p, [f["style"]] * len(p["in"])),
                         eng.compile_budget) if f["mode"] == "island"
            else grade_float(p, inputs, eng.compile_budget, amp16=DEV_CUDA))
    for pth, lab, sev, det in rows or []:
        mark = "  <<<" if pth == f["path"] else ""
        print("   %-10s %-16s sev=%-10s %s%s"
              % (pth, lab, ("inf" if sev == math.inf else "%.4f" % sev), det, mark))

def run_v4(policy=None, minutes=None, burst_sec=None, polish=None, seed=None):
    """Điểm vào dùng chung: chạy standalone (`python file.py`) hoặc từ notebook
    với Gemma đã load sẵn: run_v4(policy=GemmaProgramPolicy(model, tok))."""
    print("=" * 80)
    print("AXIOM++ v4 PROGRAM PROVER — torch %s | %s" % (torch.__version__, DEV))
    if DEV_CUDA:
        print("   GPU=%s sm_%d%d TF32=%s" % (torch.cuda.get_device_name(0),
                                             CAP_SM[0], CAP_SM[1], HAS_TF32))
        if not HAS_TF32:
            print("   (sm<80: không có TF32 → lớp precision-substitution TF32 không"
                  " thể hiện ra — im lặng ở lớp đó là ĐÚNG, không phải trượt;"
                  " bãi săn ở đây: compiled-fusion, island-autocast, amp16_theory)")
    print("=" * 80)
    phase0()
    minutes = float(os.environ.get("FUZZ_MINUTES", "8")) if minutes is None else minutes
    burst = float(os.environ.get("BURST_SEC", "180")) if burst_sec is None else burst_sec
    polish = (os.environ.get("POLISH", "1") == "1") if polish is None else polish
    eng = EngineV4(seed=int(os.environ.get("SEED", "1")) if seed is None else seed)
    fb = MutationPolicy(eng.seeds, eng.rng)
    if policy is None:
        policy = GemmaProgramPolicy.load(fb) or fb
    elif getattr(policy, "fallback", None) is None:
        policy.fallback = fb
    print("policy: %s | budget %.0f phút | burst mỗi %.0fs%s"
          % (policy.name, minutes, burst,
             "" if policy.name != "mutation" else "  (đặt GEMMA_DIR/GEMMA_ID để bật Gemma 4)"))
    t0 = time.time()
    eng.run(minutes, policy, burst_sec=burst, polish=polish)
    el = time.time() - t0
    fnds = sorted(eng.findings.values(),
                  key=lambda f: -(1e18 if f["severity"] is None else f["severity"]))
    cert = [f for f in fnds if f["label"] in (CERT, EXACTV)]
    print("-" * 80)
    print("checks=%d trong %.0fs (%.0f/s) | chương trình=%d | cells=%d | findings=%d (CERT/EXACT=%d)"
          % (eng.checks, el, eng.checks / max(el, 1e-9), len(eng.programs),
             len(eng.archive), len(fnds), len(cert)))
    for f in fnds[:15]:
        sev = "inf" if f["severity"] is None else "%.2f" % f["severity"]
        print("  %-7s %-8s %-16s sev=%-8s in=%-22s [%s]%s"
              % (f["mode"], f["path"], f["label"], sev, str(f["shapes_in"]),
                 f["known_class"],
                 " shrunk→%s" % f["shrunk_in"] if f.get("shrunk_in") else ""))
    out = Path("calm_v4_findings.json")
    out.write_text(json.dumps({"torch": torch.__version__, "device": DEV,
                               "gpu": torch.cuda.get_device_name(0) if DEV_CUDA else None,
                               "sm": list(CAP_SM), "tf32": HAS_TF32,
                               "findings": fnds}, ensure_ascii=False, indent=1),
                   encoding="utf-8")
    # CHỨNG CHỈ CONFORMANCE: sự im lặng có giá trị vì precision được chứng minh
    fam = defaultdict(lambda: {"checks": 0, "max_ratio": 0.0, "violations": 0})
    for d, (sev, _, _) in eng.archive.items():
        key = "%s|%s|%s" % (d[0], d[2], d[3])
        fam[key]["max_ratio"] = max(fam[key]["max_ratio"],
                                    0 if sev == math.inf else round(min(sev, 1e6), 4))
    for f in fnds:
        if f["label"] in (CERT, EXACTV): fam["%s|%s|%s" % (",".join(sorted(set(
            n[0] for n in f["program"]["g"]))), f["mode"], f["path"])]["violations"] += 1
    certificate = {"claim": "mọi (chương-trình, đường) đã kiểm nằm trong bao chứng minh, "
                            "trừ các violations liệt kê", "torch": torch.__version__,
                   "device": DEV, "sm": list(CAP_SM), "tf32": HAS_TF32,
                   "total_checks": eng.checks, "verdicts": dict(eng.counts),
                   "families": dict(sorted(fam.items()))}
    Path("calm_v4_certificate.json").write_text(
        json.dumps(certificate, ensure_ascii=False, indent=1), encoding="utf-8")
    print("-" * 80)
    print("Phân lớp CERT/EXACT:", dict(Counter(f["known_class"] for f in cert)))
    print("→ findings: calm_v4_findings.json | chứng chỉ: calm_v4_certificate.json")
    print("→ chạy lại 1 finding:  python %s --repro calm_v4_findings.json 0"
          % Path(sys.argv[0]).name)
    novel = [f for f in cert if f["known_class"] not in KNOWN]
    if novel:
        print("\n⚑ ỨNG VIÊN NGOÀI LỚP ĐÃ BIẾT — soi tay trước khi báo cáo:")
        for f in novel[:8]:
            print("   %s/%s %s sev=%s :: %s" % (f["mode"], f["path"], f["label"],
                                                f["severity"], f["detail"]))
    return eng

def main():
    if len(sys.argv) >= 3 and sys.argv[1] == "--repro":
        do_repro(sys.argv[2], int(sys.argv[3]) if len(sys.argv) > 3 else 0); return
    run_v4()

if __name__ == "__main__":
    main()


Writing calm_axiompp_v4_program_prover.py


In [ ]:
# === Load Gemma 4 12B (4-bit NF4) — bộ SINH CHƯƠNG TRÌNH cho máy chứng minh ====
# Chuỗi loader + cấu hình lượng tử đã chứng minh chạy được trên T4 (≈7 GB VRAM).
import os, torch

MODEL_ID = os.environ.get("GEMMA_MODEL_ID", "google/gemma-4-12B-it")
GEMMA = TOK = None
if globals().get("USE_LLM", False):
    from transformers import AutoProcessor, BitsAndBytesConfig
    import transformers as _TF
    cdt = torch.float16 if torch.cuda.get_device_capability()[0] < 8 else torch.bfloat16
    qconf = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=cdt, bnb_4bit_use_double_quant=True)
    tk = {"token": HF_TOKEN} if globals().get("HF_TOKEN") else {}
    proc = AutoProcessor.from_pretrained(MODEL_ID, **tk)
    TOK = getattr(proc, "tokenizer", proc)
    if getattr(TOK, "pad_token", None) is None and getattr(TOK, "eos_token", None):
        TOK.pad_token = TOK.eos_token
    if hasattr(TOK, "padding_side"): TOK.padding_side = "left"
    last = None
    for cn in ("AutoModelForMultimodalLM", "AutoModelForImageTextToText",
               "AutoModelForCausalLM"):
        cls = getattr(_TF, cn, None)
        if cls is None: continue
        try:
            GEMMA = cls.from_pretrained(MODEL_ID, device_map="auto", dtype="auto",
                                        low_cpu_mem_usage=True,
                                        quantization_config=qconf, **tk)
            print(">> loaded via", cn); break
        except Exception as e:
            last = e
    if GEMMA is None: raise last
    GEMMA.eval()
    try: print(">> Gemma policy VRAM: %.2f GB" % (GEMMA.get_memory_footprint() / 1024**3))
    except Exception: pass
else:
    print(">> Không có GPU/LLM → engine sẽ chạy mutation policy (vẫn ra chứng chỉ).")


processor_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.5k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/23.9G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

>> loaded via AutoModelForMultimodalLM
>> Gemma policy VRAM: 6.99 GB


In [ ]:
# === RUN: PHASE 0 tự kiểm → fuzz có chứng minh, Gemma sinh chương trình theo lô ===
import os, importlib

os.environ.setdefault("COMPILE_BUDGET", "28")   # compiled-fusion là bãi săn chính trên T4
FUZZ_MINUTES = float(os.environ.get("FUZZ_MINUTES", "25"))
BURST_SEC    = float(os.environ.get("BURST_SEC", "240"))

import calm_axiompp_v4_program_prover as axv4
importlib.reload(axv4)                          # an toàn khi sửa cell engine rồi chạy lại

policy = None
if globals().get("GEMMA") is not None:
    policy = axv4.GemmaProgramPolicy(GEMMA, TOK)   # fallback mutation tự gắn trong run_v4
eng = axv4.run_v4(policy=policy, minutes=FUZZ_MINUTES, burst_sec=BURST_SEC)

# Bằng chứng lên Drive (nếu cell rescue đã mount)
for f in ("calm_v4_findings.json", "calm_v4_certificate.json"):
    if "calm_copy_to_drive" in globals():
        p = calm_copy_to_drive(f)
        if p: print(">> Drive:", p)


AXIOM++ v4 PROGRAM PROVER — torch 2.11.0+cu128 | cuda
   GPU=Tesla T4 sm_75 TF32=False
   (sm<80: không có TF32 → lớp precision-substitution TF32 không thể hiện ra — im lặng ở lớp đó là ĐÚNG, không phải trượt; bãi săn ở đây: compiled-fusion, island-autocast, amp16_theory)
--------------------------------------------------------------------------------
PHASE 0: tự kiểm máy chứng minh (CPU, nhanh)
  (a) island bit-exact CPU: PASS
  (b) eager-trong-bao trên 38 instance: worst cert_ratio=0.1888 → PASS
  (c) sanitizer: chặn 4/5 mẫu hỏng (mẫu đầu hợp lệ thật)
  (d) autocast-trong-bao-fp16 trên 4 instance: worst=0.0062 → PASS
PHASE 0: PASS — máy chứng minh lành
policy: gemma4-program | budget 25 phút | burst mỗi 240s


W0611 09:43:15.785000 2526 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
W0611 09:43:31.570000 2526 torch/_dynamo/convert_frame.py:1743] [0/8] torch._dynamo hit config.recompile_limit (8)
W0611 09:43:31.570000 2526 torch/_dynamo/convert_frame.py:1743] [0/8]    function: 'run' (/content/calm_axiompp_v4_program_prover.py:379)
W0611 09:43:31.570000 2526 torch/_dynamo/con

  ~ burst #1 (gemma4-program): +11 chương trình, 1 lỗi cú pháp, 177.5s
  [  3730 check]    18 check/s | prog 160 | cells 597 | sile 3730 | skip 28 inv 0 | 213s
  [ 17546 check]    77 check/s | prog 725 | cells 1727 | gray 8 sile 17538 | skip 149 inv 0 | 228s
  [ 30706 check]   127 check/s | prog 1242 | cells 2450 | gray 8 sile 30698 | skip 330 inv 0 | 243s
  [ 43812 check]   170 check/s | prog 1728 | cells 3070 | gray 8 sile 43804 | skip 474 inv 0 | 258s
  [ 56143 check]   206 check/s | prog 2187 | cells 3722 | gray 12 sile 56131 | skip 653 inv 0 | 273s
  [ 68478 check]   238 check/s | prog 2607 | cells 4156 | gray 20 sile 68458 | skip 826 inv 0 | 288s
  [ 79861 check]   264 check/s | prog 3010 | cells 4544 | gray 28 sile 79833 | skip 983 inv 0 | 303s
  [ 91616 check]   288 check/s | prog 3425 | cells 4865 | gray 28 sile 91588 | skip 1134 inv 0 | 318s
  [103085 check]   310 check/s | prog 3836 | cells 5178 | gray 32 sile 103053 | skip 1260 inv 0 | 333s
  [114809 check]   330 check/s | 

In [ ]:
# %% ════════════════════════════════════════════════════════════════════════
# AXIOM++ v5 — SURPRISE + STAR: cell-chạy-tiếp sau run v4 trên T4.
# Mở rộng (KHÔNG sửa) calm_axiompp_v4_program_prover.py — import và kế thừa.
#
# Dùng trong notebook (Gemma 4 ĐÃ load ở cell trước → tái dùng, khỏi load lại):

import json, math, os, sys, time
from collections import Counter
from pathlib import Path

try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
import torch
import calm_axiompp_v4_program_prover as v4

# ── (2) mở khoá compiled NGAY khi import — tên config đổi theo đời torch ─────
def _unlock_dynamo(limit=256):
    try:
        import torch._dynamo as dyn
        hit = []
        for name in ("recompile_limit", "cache_size_limit",
                     "accumulated_recompile_limit", "accumulated_cache_size_limit"):
            if hasattr(dyn.config, name):
                try: setattr(dyn.config, name, limit); hit.append(name)
                except Exception: pass
        if hit: print(">> dynamo unlock: %s = %d" % (", ".join(hit), limit))
    except Exception as e:
        print(">> dynamo unlock bỏ qua:", str(e)[:60])
_unlock_dynamo()

# ── (1) surprise: k_eff = chiều tích-luỹ dài nhất của chương trình ───────────
def k_eff(prog):
    sh = prog["_shapes"]; k = 1
    for n in prog["_expanded"]:
        if n[0] in ("matmul", "sum", "sumk", "cumsum"):
            k = max(k, sh[n[1]][1])
    return max(k, 1)

def surprise_of(sev, prog):
    if sev == math.inf: return math.inf
    return sev * math.sqrt(k_eff(prog))

SURPRISE_GATE = 4.0          # gray có surprise vượt mức này → đáng lưu dù sev<0.92

# ── (3) seed v5: reduction-split / online-softmax / chuỗi khuếch đại ─────────
SEED_RAW_V5 = [
    # trần sanitizer v4 = 1024/chiều — warning online-softmax trên T4 đã xuất
    # hiện với chính các chương trình ≤1024, nên k≈1024 đủ chạm reduction-split;
    # jitter của engine tự nhích quanh 1023/1024/1025.
    {"in": [[32, 1024]], "g": [["softmax", 0]], "o": 1},
    {"in": [[16, 1024]], "g": [["logsumexp", 0]], "o": 1},
    {"in": [[24, 1000]], "g": [["softmax", 0], ["cumsum", 1]], "o": 2},
    {"in": [[16, 1024]], "g": [["cumsum", 0], ["cumsum", 1]], "o": 2},          # double-prefix
    {"in": [[8, 1024], [1024, 8]], "g": [["matmul", 0, 1], ["cumsum", 2]], "o": 3},
    {"in": [[64, 64], [64, 64], [64, 64]],                                       # add-chain x+=A@B
     "g": [["matmul", 0, 1], ["add", 3, 2], ["add", 4, 3], ["add", 5, 3],
           ["add", 6, 3], ["add", 7, 3]], "o": 8},
    {"in": [[48, 1024], [1024, 48]],
     "g": [["softmax", 0], ["matmul", 2, 1], ["softmax", 3]], "o": 4},          # sandwich
    {"in": [[12, 1024]], "g": [["sum", 0]], "o": 1},
    {"in": [[8, 1024], [1024, 8]], "g": [["matmul", 0, 1], ["exp", 2], ["sum", 3]], "o": 4},
]

class TaggedPolicy:
    """Bọc policy để gắn xuất xứ chương trình (gemma/mutation) cho STaR."""
    def __init__(self, inner, tag):
        self.inner, self.tag = inner, tag
        self.name = inner.name
        self.fallback = getattr(inner, "fallback", None)
    def generate(self, summary, n=6):
        progs, st = self.inner.generate(summary, n)
        for p in progs: p["_src"] = self.tag
        return progs, st

class EngineV5(v4.EngineV4):
    """Kế thừa toàn bộ máy v4; CHỈ đổi fitness archive sang surprise + gắn
    triage expected_tight + lưu surprise/k_eff/xuất-xứ vào findings.
    Phán quyết (CERT/EXACT/GRAY/SILENT) giữ nguyên định lý của v4."""
    def evaluate(self, prog, mode, style, vseed, shrink=True):
        try:
            inputs = v4.gen_inputs(prog, mode, style, vseed)
            if mode == "island":
                peakM = v4.magnitude_max(prog, [style] * len(prog["in"]))
                rows = v4.grade_island(prog, inputs, peakM, self.compile_budget)
            else:
                amp = v4.DEV_CUDA and prog["_has_mm"] and self.rng.random() < 0.5
                rows = v4.grade_float(prog, inputs, self.compile_budget, amp16=amp)
            if rows is None:
                self.skips += 1; return 0.0
        except Exception:
            self.invalid += 1; return 0.0
        best = 0.0
        ke = k_eff(prog)
        for path, lab, sev, detail in rows:
            self.checks += 1; self.counts[lab] += 1
            fit = surprise_of(sev, prog)                       # ← fitness MỚI
            d = v4.descriptor(prog, mode, path)
            cur = self.archive.get(d)
            if cur is None or fit > cur[0]:
                self.archive[d] = (fit, prog, (mode, style, vseed))
            best = max(best, 0.0 if fit == math.inf else min(fit, 1e6))
            keep = (lab in (v4.CERT, v4.EXACTV)
                    or (lab == v4.GRAY and sev > 0.92)
                    or (lab == v4.GRAY and fit > SURPRISE_GATE))
            if keep:
                fk = (prog["_key"], mode, path, lab)
                prev = self.findings.get(fk)
                sev_n = None if sev == math.inf else round(sev, 4)
                fit_n = None if fit == math.inf else round(fit, 3)
                if prev is None or (fit_n or 1e18) > (prev.get("surprise") or 0):
                    fnd = {"program": {"in": prog["in"], "g": prog["g"], "o": prog["o"]},
                           "mode": mode, "style": style, "value_seed": vseed,
                           "path": path, "label": lab, "severity": sev_n,
                           "surprise": fit_n, "k_eff": ke,
                           "expected_tight": bool(lab == v4.GRAY and ke <= 4),
                           "src": prog.get("_src", "evolve"),
                           "detail": detail, "known_class": v4.path_class(path),
                           "shapes_in": prog["in"]}
                    if shrink and lab in (v4.CERT, v4.EXACTV):
                        fnd["shrunk_in"] = self.shrink(prog, mode, style, vseed, path)
                    self.findings[fk] = fnd
        return best

# ── triage findings của run TRƯỚC (nếu có file trong thư mục) ────────────────
def triage_previous(path="calm_v4_findings.json"):
    p = Path(path)
    if not p.exists(): return
    try:
        data = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        return
    rows = data.get("findings", [])
    if not rows: return
    print("-" * 80)
    print("TRIAGE run trước (%s): %d finding" % (path, len(rows)))
    n_tight = 0
    for f in rows:
        pr, why = v4.sanitize(f["program"])
        ke = k_eff(pr) if pr else -1
        sev = f.get("severity")
        tight = (f["label"] == v4.GRAY and 0 < ke <= 4 and (sev or 0) <= 1.0)
        n_tight += tight
        sur = None if sev is None else round(sev * math.sqrt(max(ke, 1)), 2)
        print("  %-7s %-8s sev=%-6s k_eff=%-5d surprise=%-7s %s"
              % (f["path"], f["label"][:8], sev, ke, sur,
                 "EXPECTED_TIGHT (mép định lý γ_k gần đạt — không phải bug)" if tight
                 else "← giữ lại soi"))
    print("  → %d/%d là mép-định-lý k nhỏ; v5 đổi fitness sang surprise để khỏi"
          " leo lại vào đó." % (n_tight, len(rows)))

# ── STaR vòng 1: gặt elites thành SFT rows (schema pipeline gemma4_sft) ──────
STAR_INSTR = ("You steer a certified tensor-program reliability tester for deep-learning "
              "libraries (scientific-software QA). Write ONE short tensor program in the "
              "CALM DSL as a single JSON object {\"in\":[[rows,cols],...],\"g\":[[op,arg"
              "(,arg)],...],\"o\":<last node id>} that stresses the given target. Reply "
              "with ONLY the JSON.")

def star_harvest(eng, out="calm_star_round1.jsonl", cap=400):
    seen, rows = set(), []
    for d, (fit, prog, inst) in sorted(eng.archive.items(),
                                       key=lambda kv: -(kv[1][0] if kv[1][0] != math.inf
                                                        else 1e18)):
        if fit != math.inf and fit <= 1.0: break          # phần đuôi nhạt — dừng
        key = prog["_key"]
        if key in seen: continue
        seen.add(key)
        sig, depth, mode, path = d
        rows.append({
            "instruction": STAR_INSTR,
            "input": ("target: ops={%s} mode=%s path=%s | observed surprise=%.3f "
                      "k_eff=%d | prefer tile-boundary sizes (2^k±1) and accumulation "
                      "chains" % (sig, mode, path,
                                  (999.0 if fit == math.inf else fit), k_eff(prog))),
            "output": json.dumps({"in": prog["in"], "g": prog["g"], "o": prog["o"]},
                                 ensure_ascii=False),
            "meta": {"src": prog.get("_src", "evolve"),
                     "surprise": None if fit == math.inf else round(fit, 3),
                     "descriptor": list(d)},
        })
        if len(rows) >= cap: break
    Path(out).write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in rows),
                         encoding="utf-8")
    srcs = Counter(r["meta"]["src"] for r in rows)
    print("→ STaR round-1: %d mẫu SFT → %s | xuất xứ: %s" % (len(rows), out, dict(srcs)))
    return rows

# ════════════════════════════════════════════════════════════════════════════
def run(policy=None, minutes=None, burst_sec=None, seed=None):
    print("=" * 80)
    print("AXIOM++ v5 SURPRISE+STAR — torch %s | %s | fitness=cert_ratio·√k_eff"
          % (torch.__version__, v4.DEV))
    print("=" * 80)
    triage_previous()
    v4.phase0()
    minutes = float(os.environ.get("FUZZ_MINUTES", "30")) if minutes is None else minutes
    burst = float(os.environ.get("BURST_SEC", "240")) if burst_sec is None else burst_sec
    os.environ.setdefault("COMPILE_BUDGET", "64")          # limit đã mở → budget có nghĩa
    eng = EngineV5(seed=int(os.environ.get("SEED", "2")) if seed is None else seed)
    extra = []
    for raw in SEED_RAW_V5:
        p, why = v4.sanitize(raw)
        if p is None:                       # version v4 trên đĩa khác trần → bỏ mềm
            print(">> seed v5 bị sanitizer chặn (%s): %s" % (why, raw)); continue
        p["_src"] = "seed_v5"; extra.append(p); eng.programs[p["_key"]] = p
    eng.seeds = eng.seeds + extra
    fb = v4.MutationPolicy(eng.seeds, eng.rng)
    if policy is None:
        policy = v4.GemmaProgramPolicy.load(fb) or fb
    elif getattr(policy, "fallback", None) is None:
        policy.fallback = fb
    policy = TaggedPolicy(policy, "gemma" if policy.name != "mutation" else "mutation")
    print("policy: %s | budget %.0f phút | burst mỗi %.0fs | seed v5: %d chuỗi khuếch đại"
          % (policy.name, minutes, burst, len(extra)))
    t0 = time.time()
    # polish=False: chọn-elite giờ theo surprise; gradient-polish của v4 chọn theo
    # thang cũ nên tạm tắt thay vì chạy sai thang.
    eng.run(minutes, policy, burst_sec=burst, polish=False)
    el = time.time() - t0

    fnds = sorted(eng.findings.values(),
                  key=lambda f: -(1e18 if f["surprise"] is None else f["surprise"]))
    cert = [f for f in fnds if f["label"] in (v4.CERT, v4.EXACTV)]
    shown = [f for f in fnds if not f.get("expected_tight")]
    print("-" * 80)
    print("checks=%d trong %.0fs (%.0f/s) | chương trình=%d | cells=%d | findings=%d"
          " (CERT/EXACT=%d, expected_tight=%d đã lọc)"
          % (eng.checks, el, eng.checks / max(el, 1e-9), len(eng.programs),
             len(eng.archive), len(fnds), len(cert),
             sum(f.get("expected_tight", False) for f in fnds)))
    for f in shown[:15]:
        sur = "inf" if f["surprise"] is None else "%.2f" % f["surprise"]
        sev = "inf" if f["severity"] is None else "%.2f" % f["severity"]
        print("  %-7s %-12s %-16s surprise=%-8s sev=%-6s k=%-5d src=%-8s in=%s"
              % (f["mode"], f["path"], f["label"], sur, sev, f["k_eff"], f["src"],
                 f["shapes_in"]))
    Path("calm_v5_findings.json").write_text(
        json.dumps({"torch": torch.__version__, "device": v4.DEV,
                    "fitness": "surprise=cert_ratio*sqrt(k_eff)",
                    "findings": fnds}, ensure_ascii=False, indent=1), encoding="utf-8")
    arch = [{"descriptor": list(d), "surprise": (None if s == math.inf else round(s, 3)),
             "program": {"in": pr["in"], "g": pr["g"], "o": pr["o"]},
             "src": pr.get("_src", "evolve"), "instance": list(inst)}
            for d, (s, pr, inst) in sorted(eng.archive.items(),
                                           key=lambda kv: -(kv[1][0] if kv[1][0] != math.inf
                                                            else 1e18))[:1200]]
    Path("calm_v5_archive.json").write_text(
        json.dumps(arch, ensure_ascii=False, indent=1), encoding="utf-8")
    cert_doc = {"claim": "mọi (chương-trình, đường) đã kiểm nằm trong bao chứng minh, "
                         "trừ violations; gray xếp theo surprise, mép-định-lý đã tự lọc",
                "torch": torch.__version__, "device": v4.DEV,
                "total_checks": eng.checks, "verdicts": dict(eng.counts),
                "expected_tight_filtered": sum(f.get("expected_tight", False) for f in fnds)}
    Path("calm_v5_certificate.json").write_text(
        json.dumps(cert_doc, ensure_ascii=False, indent=1), encoding="utf-8")
    star_harvest(eng)
    print("→ calm_v5_findings.json | calm_v5_archive.json | calm_v5_certificate.json")
    print("Phân lớp CERT/EXACT:", dict(Counter(f["known_class"] for f in cert)) or "{} "
          "(im trên release = đúng; surprise-gray + STaR là sản phẩm của vòng này)")
    return eng

def main():
    run()

if __name__ == "__main__":
    main()


>> dynamo unlock: recompile_limit, cache_size_limit, accumulated_recompile_limit, accumulated_cache_size_limit = 256
AXIOM++ v5 SURPRISE+STAR — torch 2.11.0+cu128 | cuda | fitness=cert_ratio·√k_eff
--------------------------------------------------------------------------------
TRIAGE run trước (calm_v4_findings.json): 10 finding
  eager   GRAY_SUS sev=0.9677 k_eff=2     surprise=1.37    EXPECTED_TIGHT (mép định lý γ_k gần đạt — không phải bug)
  vs_cpu  GRAY_SUS sev=0.9677 k_eff=2     surprise=1.37    EXPECTED_TIGHT (mép định lý γ_k gần đạt — không phải bug)
  layout  GRAY_SUS sev=0.9677 k_eff=2     surprise=1.37    EXPECTED_TIGHT (mép định lý γ_k gần đạt — không phải bug)
  splitk  GRAY_SUS sev=0.9677 k_eff=2     surprise=1.37    EXPECTED_TIGHT (mép định lý γ_k gần đạt — không phải bug)
  eager   GRAY_SUS sev=0.9366 k_eff=2     surprise=1.32    EXPECTED_TIGHT (mép định lý γ_k gần đạt — không phải bug)
  vs_cpu  GRAY_SUS sev=0.9366 k_eff=2     surprise=1.32    EXPECTED_TIGHT (mép định

KeyboardInterrupt: 

In [ ]:
# %% ════════════════════════════════════════════════════════════════════════
# AXIOM++ v6 — FAULT-LAB: đo ĐỘ NHẠY của máy chứng minh bằng lỗi cấy chủ động
# (fault injection / mutation-testing-of-the-oracle — chuẩn đánh giá detector
# khi bug thật trên release hiếm). Chạy ~3-5 phút trên T4, ra bảng + JSON.
#
# Notebook (sau cell engine v4):
#   import calm_axiompp_v6_faultlab as v6; v6.run()
# Standalone:  python calm_axiompp_v6_faultlab.py
#
# ════════════════════════════════════════════════════════════════════════════
import json, math, sys, time
from pathlib import Path

try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass
import torch
import calm_axiompp_v4_program_prover as v4

TOL_BASELINE = 1e-4          # đúng global tol notebook cũ học từ 60 cặp lành

def trunc_mantissa(x, keep_bits):
    """Cắt mantissa fp32 còn keep_bits (round-toward-zero ⇒ bias hệ thống —
    mô phỏng thay-độ-chính-xác-âm-thầm kiểu TF32(10)/bf16(7))."""
    drop = 23 - keep_bits
    xi = x.contiguous().view(torch.int32)
    mask = torch.tensor(-(1 << drop), dtype=torch.int32, device=x.device)
    return (xi & mask).view(torch.float32)

DEFECTS = {
    # name -> (fA, fB, fC): biến đổi input A, B và output C của MỌI matmul
    "tf32emu_10bit": (lambda t: trunc_mantissa(t, 10), lambda t: trunc_mantissa(t, 10), None),
    "bf16emu_7bit":  (lambda t: trunc_mantissa(t, 7),  lambda t: trunc_mantissa(t, 7),  None),
    "fp16_sub":      (lambda t: t.half().float(), lambda t: t.half().float(),
                      lambda t: t.half().float()),
    # bias làm-tròn-hệ-thống: 4 ulp nằm TRONG quyền-làm-tròn hợp lệ (γ_k) →
    # oracle ĐÚNG ĐẮN phải im (sàn chứng-minh-được); 128 ulp vượt γ_k khi k nhỏ
    # → lộ rõ BIÊN GIỚI: bắt được ⟺ bias > γ_k.
    "bias_4ulp":     (None, None, lambda t: t * (1.0 + 2.0 ** -22)),
    "bias_128ulp":   (None, None, lambda t: t * (1.0 + 2.0 ** -16)),
    # đối chứng sạch — PHẢI im
    "control_eager":  (None, None, None),
    "control_splitk": "SPLITK",
}
SUBSTITUTION = ("tf32emu_10bit", "bf16emu_7bit", "fp16_sub")

def make_defect_runner(prog, defect):
    """Runner fp32 với defect cấy vào matmul; defect='SPLITK' = reassociation lành."""
    expanded, n_in = prog["_expanded"], len(prog["in"])
    if defect == "SPLITK":
        return v4.make_runner(expanded, n_in, splitk=True)
    fA, fB, fC = defect
    def run(*xs):
        vals = list(xs)
        for n in expanded:
            op, a = n[0], n[1]
            b = n[2] if len(n) > 2 else 0
            if op == "matmul":
                A = fA(vals[a]) if fA else vals[a]
                B = fB(vals[b]) if fB else vals[b]
                y = A @ B
                if fC: y = fC(y)
            else:
                y = v4._apply(op, vals, a, b)
            vals.append(y)
        return vals[-1]
    return run

PROGS = [
    ("matmul_k64",   {"in": [[64, 64], [64, 64]],     "g": [["matmul", 0, 1]], "o": 2}),
    ("matmul_k256",  {"in": [[64, 256], [256, 64]],   "g": [["matmul", 0, 1]], "o": 2}),
    ("matmul_k1024", {"in": [[32, 1024], [1024, 32]], "g": [["matmul", 0, 1]], "o": 2}),
    ("mm_relu_mm",   {"in": [[64, 128], [128, 64], [64, 48]],
                      "g": [["matmul", 0, 1], ["relu", 3], ["matmul", 4, 2]], "o": 5}),
    ("mm_softmax",   {"in": [[48, 256], [256, 48]],
                      "g": [["matmul", 0, 1], ["softmax", 2]], "o": 3}),
]
SCALES = [("scale_0.5", 0.5), ("scale_0.05", 0.05)]   # nhỏ → baseline abs-tol mù

def run(out_json="calm_v6_faultlab.json"):
    print("=" * 80)
    print("AXIOM++ v6 FAULT-LAB — torch %s | %s | baseline tol=%.0e (oracle cũ)"
          % (torch.__version__, v4.DEV, TOL_BASELINE))
    print("=" * 80)
    torch.manual_seed(7)
    rows = []
    t0 = time.time()
    for pname, raw in PROGS:
        prog, why = v4.sanitize(raw)
        assert prog is not None, (pname, why)
        for sname, sc in SCALES:
            inputs = [torch.randn(*s, device=v4.DEV) * sc for s in
                      (tuple(x) for x in prog["in"])]
            ins64 = [x.double() for x in inputs]
            ref, E, V = v4.envelope_eval(prog, ins64)
            if not V.any():
                continue
            for dname, defect in DEFECTS.items():
                try:
                    obs = make_defect_runner(prog, defect)(*inputs).double()
                except Exception as e:
                    rows.append({"prog": pname, "scale": sname, "defect": dname,
                                 "error": str(e)[:60]}); continue
                d = torch.where(V, (obs - ref).abs(), torch.zeros_like(ref))
                sev = float((d / E.clamp_min(v4.DENORM_SLACK)).max())
                lab = (v4.CERT if sev > 1.0 else
                       (v4.GRAY if sev > v4.TAU_GRAY else v4.SILENT))
                dmax = float(d.max())
                rel = dmax / max(float(ref.abs().max()), 1e-300)
                base_caught = (dmax > TOL_BASELINE) or (rel > TOL_BASELINE)
                rows.append({"prog": pname, "scale": sname, "defect": dname,
                             "sev": round(sev, 3), "label": lab,
                             "max_abs_diff": float("%.3e" % dmax),
                             "baseline_tol_caught": bool(base_caught)})
    el = time.time() - t0

    # ── bảng + chấm điểm ────────────────────────────────────────────────────
    inj = [r for r in rows if "control" not in r["defect"] and "error" not in r]
    ctl = [r for r in rows if "control" in r["defect"] and "error" not in r]
    sub = [r for r in inj if r["defect"] in SUBSTITUTION]
    bias = [r for r in inj if r["defect"].startswith("bias")]
    fp_ctl = sum(r["label"] != v4.SILENT for r in ctl)
    print("%-13s %-11s %-15s %10s  %-9s %-12s %s"
          % ("prog", "scale", "defect", "sev", "verdict", "max|Δ|", "baseline tol bắt?"))
    for r in inj + ctl:
        base = "—" if "control" in r["defect"] else \
               ("CÓ" if r["baseline_tol_caught"] else "**LỌT**")
        print("%-13s %-11s %-15s %10.3g  %-9s %-12.3e %s"
              % (r["prog"], r["scale"], r["defect"], r["sev"], r["label"][:9],
                 r["max_abs_diff"], base))
    sub_cert = sum(r["label"] == v4.CERT for r in sub)
    sub_gray = sum(r["label"] == v4.GRAY for r in sub)
    sub_base = sum(r["baseline_tol_caught"] for r in sub)
    blind = sum((not r["baseline_tol_caught"]) and r["label"] == v4.CERT for r in sub)
    print("-" * 80)
    print("[A] THAY-ĐỘ-CHÍNH-XÁC-ÂM-THẦM (lớp bug thật, kiểu TF32/bf16/fp16):")
    print("    prover: CERT %d/%d + GRAY %d (phát hiện %.0f%%) | baseline tol: %d/%d"
          % (sub_cert, len(sub), sub_gray,
             100.0 * (sub_cert + sub_gray) / max(len(sub), 1), sub_base, len(sub)))
    print("    trong đó %d ca Δ quá nhỏ baseline MÙ (Δ<1e-4) nhưng prover vẫn CERT"
          % blind)
    print("[B] BIAS LÀM-TRÒN-HỆ-THỐNG — biên giới chứng-minh-được (bắt ⟺ bias>γ_k):")
    print("    (im ở bias≤γ_k là ĐÚNG ĐẮN: trong quyền-làm-tròn hợp lệ, mọi oracle")
    print("     sound phải im — lớp đó v5 săn bằng chuỗi khuếch đại + surprise)")
    bsum = {}
    for r in bias:
        key = (r["defect"], r["prog"].split("_")[-1] if "k" in r["prog"] else r["prog"])
        cur = bsum.get(key)
        if cur is None or r["sev"] > cur: bsum[key] = r["sev"]
    for (dn, pk), sv in sorted(bsum.items()):
        print("    %-11s %-13s sev=%-8.3g %s" % (dn, pk, sv,
              "CERT (vượt γ_k)" if sv > 1 else "im đúng (≤γ_k)"))
    print("[C] ĐỐI CHỨNG SẠCH: %d/%d im lặng (báo nhầm = %d) — đo trong CÙNG thí nghiệm"
          % (len(ctl) - fp_ctl, len(ctl), fp_ctl))
    print("(%.1fs)" % el)
    Path(out_json).write_text(json.dumps(
        {"torch": torch.__version__, "device": v4.DEV, "baseline_tol": TOL_BASELINE,
         "substitution_total": len(sub), "substitution_cert": sub_cert,
         "substitution_gray": sub_gray, "substitution_baseline": sub_base,
         "baseline_blind_prover_cert": blind,
         "controls_total": len(ctl), "controls_false_positive": fp_ctl,
         "rows": rows}, ensure_ascii=False, indent=1), encoding="utf-8")
    print("→", out_json)
    return rows

if __name__ == "__main__":
    run()


AXIOM++ v6 FAULT-LAB — torch 2.11.0+cu128 | cuda | baseline tol=1e-04 (oracle cũ)
prog          scale       defect                 sev  verdict   max|Δ|       baseline tol bắt?
matmul_k64    scale_0.5   tf32emu_10bit          151  CERTIFIED 8.074e-03    CÓ
matmul_k64    scale_0.5   bf16emu_7bit      1.11e+03  CERTIFIED 6.658e-02    CÓ
matmul_k64    scale_0.5   fp16_sub              68.3  CERTIFIED 2.884e-03    CÓ
matmul_k64    scale_0.5   bias_4ulp            0.051  SILENT    2.920e-06    **LỌT**
matmul_k64    scale_0.5   bias_128ulp           2.75  CERTIFIED 1.650e-04    CÓ
matmul_k64    scale_0.05  tf32emu_10bit          128  CERTIFIED 5.892e-05    CÓ
matmul_k64    scale_0.05  bf16emu_7bit      1.08e+03  CERTIFIED 5.391e-04    CÓ
matmul_k64    scale_0.05  fp16_sub              83.3  CERTIFIED 3.804e-05    CÓ
matmul_k64    scale_0.05  bias_4ulp            0.057  SILENT    2.274e-08    **LỌT**
matmul_k64    scale_0.05  bias_128ulp           2.76  CERTIFIED 1.273e-06    **LỌT**
matmul_k